# Model Comparison Notebook

This notebook will compare the baseline logistic regression model against a random forest classifier using training-data cross-validation. The original data prep and train/test split are reproduced to keep it consistent. The held-out test set is not used for model selection or threshold impact.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/creditsense_applications.csv")

print("Original dataset shape:", df.shape)

Importing the dataset and confirming it is the expected shape.

In [ ]:
df_model = df.dropna(
    subset=["savings_balance_gbp", "years_at_current_address"]
).copy()

print("Modelling dataset shape:", df_model.shape)
print("Rows excluded:", len(df) - len(df_model))

Removing the known missing data and confirming only those have been removed.

In [ ]:
excluded_features = [
    "application_id",
    "default_within_24m",
    "age_years",
    "sex",
    "marital_status",
    "residency_status",
    "num_dependents",
    "region",
    "loan_purpose",
    "application_channel",
    "previous_application_declined"
]

Duplicating the `excluded_features` list used in my baseline model.

In [ ]:
X = df_model.drop(columns=excluded_features)

y = df_model["default_within_24m"]

print("predictive features:", X.shape[1])
print("Target:", y.name)
print("\nSelected features:")
print(X.columns.tolist())

Duplicating the predictive features list and ensuring it has been successful.

In [ ]:
nominal_features = [
    "employment_status",
    "housing_status",
    "credit_history",
    "checking_account_status"
]

binary_features = [
    "has_savings_account",
    "has_guarantor"
]

numerical_features = [
    col for col in X.columns
    if col not in nominal_features + binary_features
]

Recreating the original feature split in order to define how I handle each.

In [ ]:
print("Nominal:", nominal_features)
print("Binary:", binary_features)
print("Numerical:", numerical_features)

print("\nTotal features:",
      len(nominal_features)
      + len(binary_features)
      + len(numerical_features))

Confirming the split has been successful.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Ran exactly the same split as in my baseline logistic regression model.

In [ ]:
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining default rate:")
print(y_train.value_counts(normalize=True))

print("\nTesting default rate:")
print(y_test.value_counts(normalize=True))

Ensuring the split has been successful.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

Imported all 3 transformers ready for comparison.

In [ ]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_features),
        ("bin", OrdinalEncoder(
            categories=[["No", "Yes"], ["No", "Yes"]]
        ), binary_features)
    ]
)

Took the preprocessing method from the baseline model and simply renamed and adjusted the numerical method. I chose to not use scaling on the numerical features in this model as it will respond better for the decision tree.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

Import the pipeline and model type.

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        ("classifier", RandomForestClassifier(
            random_state=42
        ))
    ]
)

Created the model pipeline but have not fitted it to any data yet. Also important to note that as random forests use a different method entirely we have used different training hyperparameters.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

Recreated the cross-validation configuration from my baseline.

In [ ]:
rf_cv_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("ROC_AUC per fold:", rf_cv_scores)
print("Mean ROC_AUC:", rf_cv_scores.mean())
print("Standard deviation:", rf_cv_scores.std())

Evaluation of the random forest shows worse scores upon initial inspection compared with logistic regression. Also note we have not used `.fit()` yet as when using cross-validation it is doing it naturally in isolation within each fold.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_features),
        ("bin", OrdinalEncoder(
            categories=[["No", "Yes"], ["No", "Yes"]]
        ), binary_features)
    ]
)

Exact copy of preprocessor cell from baseline model.

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline(
    steps =[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

Exact copy of the model cell in the baseline but added the import for `LogisticRegression`.

In [ ]:
cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("ROC_AUC per fold:", cv_scores)
print("Mean ROAC_AUC:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

Exact copy of the `cross_val_scores` section from the baseline model in order to get a comparison.

In [ ]:
from sklearn.model_selection import cross_val_predict

Importing the cross validation prediction method in order to determine tangible results for our stakeholder.

In [ ]:
lr_cv_predictions = cross_val_predict(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    method="predict"
)

Generating one class prediction for each applicant remaining using the cross validation fold data for logistic regression.

In [ ]:
rf_cv_predictions = cross_val_predict(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict"
)

Exactly the same but for random forests.

In [ ]:
from sklearn.metrics import confusion_matrix

lr_cm = confusion_matrix(y_train, lr_cv_predictions)
rf_cm = confusion_matrix(y_train, rf_cv_predictions)

print("Logistic Regression:")
print(lr_cm)

print("Random Forest:")
print(rf_cm)

Evidence that Logistic Regression has a stronger default recall at the default threshold in comparison to Random Forests.